# Chroma CRUD Operations

This notebook walks through create, read, update, and delete operations with a local Chroma vector store.

In [1]:
import os
import shutil
from pathlib import Path
from uuid import uuid4
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings

# 1. Set Up Paths and the Vector Store

In [2]:
# Resolve the project root so the notebook works from either the repo root or the notebooks folder.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

project_root

PosixPath('/Users/rahulprajapati/Documents/GitHub/Advanced-RAG-2026/vector_stores')

In [3]:
# Load environment variables from the local .env file.
dotenv_path = project_root / ".env"
load_dotenv(dotenv_path=dotenv_path)

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Please add your OPENAI_API_KEY to the .env file before running this notebook.")

print(f"Loaded environment from: {dotenv_path}")

Loaded environment from: /Users/rahulprajapati/Documents/GitHub/Advanced-RAG-2026/vector_stores/.env


In [4]:
# Use a fixed collection name and persistence path so each rerun is predictable.
collection_name = "demo_1"
persist_directory = project_root / "notebooks" / "chroma_langchain_db"

print(f"Collection name: {collection_name}")
print(f"Persist directory: {persist_directory}")

Collection name: demo_1
Persist directory: /Users/rahulprajapati/Documents/GitHub/Advanced-RAG-2026/vector_stores/notebooks/chroma_langchain_db


In [5]:
# Start fresh so the CRUD flow produces the same result each time.
if persist_directory.exists():
    shutil.rmtree(persist_directory)
    print("Removed the old Chroma directory.")
else:
    print("No previous Chroma directory was found.")

Removed the old Chroma directory.


In [6]:
# Create the embedding model and connect it to a persistent Chroma store.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vector_store = Chroma(
    collection_name=collection_name,
    embedding_function=embeddings,
    persist_directory=str(persist_directory),
)

print("Vector store is ready.")

Vector store is ready.


## 2. Add Small Helper Functions

In [7]:
def preview_text(text, limit=80):
    """Return a short preview for cleaner notebook output."""
    if len(text) <= limit:
        return text
    return text[:limit] + "..."


def print_documents(title, docs):
    """Print Document objects in a beginner-friendly format."""
    print(title)
    for index, doc in enumerate(docs, start=1):
        print(f"{index}. id={doc.id}")
        print(f"   topic={doc.metadata.get('topic')} | doc_number={doc.metadata.get('doc_number')}")
        print(f"   content={doc.page_content}")
    print()

## 3. Create and Insert Example Documents

In [8]:
# Keep the raw sample data separate from the Document objects so it is easier to read.
document_examples = [
    {
        "topic": "AI",
        "doc_number": 1,
        "text": "Artificial intelligence helps machines perform tasks that usually need human reasoning.",
    },
    {
        "topic": "AI",
        "doc_number": 2,
        "text": "AI systems can analyze patterns in data to support predictions and automation.",
    },
    {
        "topic": "AI",
        "doc_number": 3,
        "text": "Responsible AI development includes fairness, transparency, and safety checks.",
    },
    {
        "topic": "RAG",
        "doc_number": 4,
        "text": "RAG combines retrieval with generation so the model can answer using external knowledge.",
    },
    {
        "topic": "RAG",
        "doc_number": 5,
        "text": "A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.",
    },
    {
        "topic": "RAG",
        "doc_number": 6,
        "text": "Vector stores are important in RAG because they make semantic search over embedded documents possible.",
    },
    {
        "topic": "LLM",
        "doc_number": 7,
        "text": "LLMs generate text by predicting likely next tokens from patterns learned during training.",
    },
    {
        "topic": "LLM",
        "doc_number": 8,
        "text": "Prompt design can improve how clearly an LLM follows instructions and returns useful answers.",
    },
    {
        "topic": "Cricket",
        "doc_number": 9,
        "text": "Cricket teams score runs through batting partnerships, boundaries, and quick running between the wickets.",
    },
    {
        "topic": "Cricket",
        "doc_number": 10,
        "text": "A cricket bowler can pressure batters with pace, swing, spin, and accurate line and length.",
    },
]

print(f"Prepared {len(document_examples)} document examples.")


Prepared 10 document examples.


In [9]:
for doc in document_examples:
    print(doc)
    print()

{'topic': 'AI', 'doc_number': 1, 'text': 'Artificial intelligence helps machines perform tasks that usually need human reasoning.'}

{'topic': 'AI', 'doc_number': 2, 'text': 'AI systems can analyze patterns in data to support predictions and automation.'}

{'topic': 'AI', 'doc_number': 3, 'text': 'Responsible AI development includes fairness, transparency, and safety checks.'}

{'topic': 'RAG', 'doc_number': 4, 'text': 'RAG combines retrieval with generation so the model can answer using external knowledge.'}

{'topic': 'RAG', 'doc_number': 5, 'text': 'A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.'}

{'topic': 'RAG', 'doc_number': 6, 'text': 'Vector stores are important in RAG because they make semantic search over embedded documents possible.'}

{'topic': 'LLM', 'doc_number': 7, 'text': 'LLMs generate text by predicting likely next tokens from patterns learned during training.'}

{'topic': 'LLM', 'doc_number': 8, 'text': 'Prompt des

In [10]:
# Convert the sample data into LangChain Document objects.

documents = [
    Document(
        id = str(uuid4()),
        page_content = item['text'],
        metadata = {"topic": item['topic'], "doc_number": item['doc_number']}
    )
    for item in document_examples
]


print_documents("Dummy documents prepared:", documents)


Dummy documents prepared:
1. id=384cf3e5-bab8-4b43-92f6-0d965be9fd71
   topic=AI | doc_number=1
   content=Artificial intelligence helps machines perform tasks that usually need human reasoning.
2. id=b30f76ef-f6b3-42f6-a49f-aae9be25f336
   topic=AI | doc_number=2
   content=AI systems can analyze patterns in data to support predictions and automation.
3. id=8b4b5b54-9b70-4301-9023-7d62d98f789b
   topic=AI | doc_number=3
   content=Responsible AI development includes fairness, transparency, and safety checks.
4. id=4439807e-3217-4d16-8e22-3e5cda6b1ef6
   topic=RAG | doc_number=4
   content=RAG combines retrieval with generation so the model can answer using external knowledge.
5. id=a80fd979-8d37-4c27-9fd8-fb52189ab165
   topic=RAG | doc_number=5
   content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.
6. id=b5678f4f-6ceb-4e84-a385-32f5e661f2ad
   topic=RAG | doc_number=6
   content=Vector stores are important in RAG because they mak

In [11]:
documents[0].id

'384cf3e5-bab8-4b43-92f6-0d965be9fd71'

In [12]:
# Insert the documents into Chroma. Chroma creates embeddings during this step.
document_ids = vector_store.add_documents(documents)

print("Inserted document ids:")
for doc_id in document_ids:
    print(doc_id)

print(f"\nTotal inserted documents: {len(document_ids)}")

Inserted document ids:
384cf3e5-bab8-4b43-92f6-0d965be9fd71
b30f76ef-f6b3-42f6-a49f-aae9be25f336
8b4b5b54-9b70-4301-9023-7d62d98f789b
4439807e-3217-4d16-8e22-3e5cda6b1ef6
a80fd979-8d37-4c27-9fd8-fb52189ab165
b5678f4f-6ceb-4e84-a385-32f5e661f2ad
dfd3df89-ce06-4c4d-b384-be3a38e46b6d
b99df7ac-cbf3-4a77-acdd-d60f38e3b3c1
5583866f-b4e6-47cd-9e36-2c559d60bbc3
1c93dbe4-88de-41df-ad8a-935ec3152f09

Total inserted documents: 10


## 4. Read the Stored Data Back

In [16]:
raw_records = vector_store.get()
print(raw_records.keys())

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])


In [18]:
raw_records = vector_store.get(include=['embeddings', 'documents', 'metadatas'])
raw_records

{'ids': ['384cf3e5-bab8-4b43-92f6-0d965be9fd71',
  'b30f76ef-f6b3-42f6-a49f-aae9be25f336',
  '8b4b5b54-9b70-4301-9023-7d62d98f789b',
  '4439807e-3217-4d16-8e22-3e5cda6b1ef6',
  'a80fd979-8d37-4c27-9fd8-fb52189ab165',
  'b5678f4f-6ceb-4e84-a385-32f5e661f2ad',
  'dfd3df89-ce06-4c4d-b384-be3a38e46b6d',
  'b99df7ac-cbf3-4a77-acdd-d60f38e3b3c1',
  '5583866f-b4e6-47cd-9e36-2c559d60bbc3',
  '1c93dbe4-88de-41df-ad8a-935ec3152f09'],
 'embeddings': array([[ 0.00492096,  0.02096558,  0.01657104, ...,  0.00069427,
         -0.01638794,  0.02790833],
        [-0.01448822, -0.00543213,  0.03034973, ..., -0.02857971,
         -0.00191593,  0.04275513],
        [ 0.02270508,  0.01531982,  0.04547119, ...,  0.02301025,
          0.00806427, -0.01585388],
        ...,
        [ 0.00801086,  0.02314758,  0.02171326, ..., -0.02072144,
         -0.0178833 ,  0.01332855],
        [ 0.00882721,  0.06268311,  0.09967041, ..., -0.01583862,
         -0.01374054,  0.0368042 ],
        [ 0.01806641,  0.08551025, 

In [13]:
# The get() method returns the low-level Chroma record structure.
raw_records = vector_store.get(include=["embeddings", "metadatas", "documents"])
raw_records.keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])

In [20]:
# Pick a few ids so we can read them back in a higher-level format.
selected_ids = document_ids[-3:]
selected_ids

['b99df7ac-cbf3-4a77-acdd-d60f38e3b3c1',
 '5583866f-b4e6-47cd-9e36-2c559d60bbc3',
 '1c93dbe4-88de-41df-ad8a-935ec3152f09']

In [21]:
# get_by_ids() returns LangChain Document objects instead of the raw Chroma dictionary.
selected_documents = vector_store.get_by_ids(selected_ids)
print_documents("Documents fetched with get_by_ids():", selected_documents)

Documents fetched with get_by_ids():
1. id=b99df7ac-cbf3-4a77-acdd-d60f38e3b3c1
   topic=LLM | doc_number=8
   content=Prompt design can improve how clearly an LLM follows instructions and returns useful answers.
2. id=5583866f-b4e6-47cd-9e36-2c559d60bbc3
   topic=Cricket | doc_number=9
   content=Cricket teams score runs through batting partnerships, boundaries, and quick running between the wickets.
3. id=1c93dbe4-88de-41df-ad8a-935ec3152f09
   topic=Cricket | doc_number=10
   content=A cricket bowler can pressure batters with pace, swing, spin, and accurate line and length.



## 5. Run a Similarity Search

In [22]:
query = "How does RAG help an LLM answer questions using outside knowledge?"
query

'How does RAG help an LLM answer questions using outside knowledge?'

In [23]:
search_results = vector_store.similarity_search(query, k=3)
print(f"Query: {query}\n")
print_documents("Similarity search results:", search_results)

Query: How does RAG help an LLM answer questions using outside knowledge?

Similarity search results:
1. id=4439807e-3217-4d16-8e22-3e5cda6b1ef6
   topic=RAG | doc_number=4
   content=RAG combines retrieval with generation so the model can answer using external knowledge.
2. id=b99df7ac-cbf3-4a77-acdd-d60f38e3b3c1
   topic=LLM | doc_number=8
   content=Prompt design can improve how clearly an LLM follows instructions and returns useful answers.
3. id=a80fd979-8d37-4c27-9fd8-fb52189ab165
   topic=RAG | doc_number=5
   content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.



In [24]:
search_results

[Document(id='4439807e-3217-4d16-8e22-3e5cda6b1ef6', metadata={'topic': 'RAG', 'doc_number': 4}, page_content='RAG combines retrieval with generation so the model can answer using external knowledge.'),
 Document(id='b99df7ac-cbf3-4a77-acdd-d60f38e3b3c1', metadata={'doc_number': 8, 'topic': 'LLM'}, page_content='Prompt design can improve how clearly an LLM follows instructions and returns useful answers.'),
 Document(id='a80fd979-8d37-4c27-9fd8-fb52189ab165', metadata={'doc_number': 5, 'topic': 'RAG'}, page_content='A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.')]

In [25]:
vector_store.similarity_search_with_score(query=query, k=4)

[(Document(id='4439807e-3217-4d16-8e22-3e5cda6b1ef6', metadata={'doc_number': 4, 'topic': 'RAG'}, page_content='RAG combines retrieval with generation so the model can answer using external knowledge.'),
  0.8068076372146606),
 (Document(id='b99df7ac-cbf3-4a77-acdd-d60f38e3b3c1', metadata={'doc_number': 8, 'topic': 'LLM'}, page_content='Prompt design can improve how clearly an LLM follows instructions and returns useful answers.'),
  0.9390738010406494),
 (Document(id='a80fd979-8d37-4c27-9fd8-fb52189ab165', metadata={'doc_number': 5, 'topic': 'RAG'}, page_content='A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.'),
  1.0749804973602295),
 (Document(id='dfd3df89-ce06-4c4d-b384-be3a38e46b6d', metadata={'topic': 'LLM', 'doc_number': 7}, page_content='LLMs generate text by predicting likely next tokens from patterns learned during training.'),
  1.1317026615142822)]

## 6. Update Existing Documents

In [26]:
# We will update one RAG document and one LLM document.
ids_to_update = [document_ids[3], document_ids[7]]
ids_to_update

['4439807e-3217-4d16-8e22-3e5cda6b1ef6',
 'b99df7ac-cbf3-4a77-acdd-d60f38e3b3c1']

In [27]:
# Keep the replacement text separate so the update step stays easy to follow.
updated_examples = [
    {
        "id": ids_to_update[0],
        "topic": "RAG",
        "doc_number": 4,
        "text": "RAG improves answer quality by retrieving relevant context before the language model generates a response.",
    },
    {
        "id": ids_to_update[1],
        "topic": "LLM",
        "doc_number": 8,
        "text": "Well-written prompts help an LLM stay focused, follow instructions, and produce more reliable outputs.",
    },
]

updated_documents = [
    Document(
        id=item["id"],
        page_content=item["text"],
        metadata={"topic": item["topic"], "doc_number": item["doc_number"]},
    )
    for item in updated_examples
]

print_documents("Updated document content:", updated_documents)

Updated document content:
1. id=4439807e-3217-4d16-8e22-3e5cda6b1ef6
   topic=RAG | doc_number=4
   content=RAG improves answer quality by retrieving relevant context before the language model generates a response.
2. id=b99df7ac-cbf3-4a77-acdd-d60f38e3b3c1
   topic=LLM | doc_number=8
   content=Well-written prompts help an LLM stay focused, follow instructions, and produce more reliable outputs.



In [28]:
print([doc.page_content for doc in documents if doc.id in ids_to_update])

['RAG combines retrieval with generation so the model can answer using external knowledge.', 'Prompt design can improve how clearly an LLM follows instructions and returns useful answers.']


In [29]:
vector_store.update_documents(ids=ids_to_update, documents=updated_documents)

print("Updated these ids:")
for doc_id in ids_to_update:
    print(doc_id)

Updated these ids:
4439807e-3217-4d16-8e22-3e5cda6b1ef6
b99df7ac-cbf3-4a77-acdd-d60f38e3b3c1


In [30]:
# Read the updated records back from Chroma to confirm the new values were stored.
updated_raw_records = vector_store.get(ids=ids_to_update)

print("Raw records returned by get(ids=ids_to_update):")
for doc_id, document_text, metadata in zip(
    updated_raw_records["ids"],
    updated_raw_records["documents"],
    updated_raw_records["metadatas"],
):
    print(f"id={doc_id}")
    print(f"metadata={metadata}")
    print(f"content={preview_text(document_text)}")
    print()

Raw records returned by get(ids=ids_to_update):
id=4439807e-3217-4d16-8e22-3e5cda6b1ef6
metadata={'topic': 'RAG', 'doc_number': 4}
content=RAG improves answer quality by retrieving relevant context before the language m...

id=b99df7ac-cbf3-4a77-acdd-d60f38e3b3c1
metadata={'topic': 'LLM', 'doc_number': 8}
content=Well-written prompts help an LLM stay focused, follow instructions, and produce ...



In [31]:
updated_query = "How can retrieved context improve an LLM response in RAG?"
updated_query

'How can retrieved context improve an LLM response in RAG?'

In [32]:
updated_search_results = vector_store.similarity_search(updated_query, k=2)
print(f"Updated query: {updated_query}\n")
print_documents("Similarity search after update:", updated_search_results)

Updated query: How can retrieved context improve an LLM response in RAG?

Similarity search after update:
1. id=4439807e-3217-4d16-8e22-3e5cda6b1ef6
   topic=RAG | doc_number=4
   content=RAG improves answer quality by retrieving relevant context before the language model generates a response.
2. id=a80fd979-8d37-4c27-9fd8-fb52189ab165
   topic=RAG | doc_number=5
   content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.



## 7. Delete Documents

In [33]:
# Delete the two cricket examples so the final collection is smaller.
ids_to_delete = [document_ids[8], document_ids[9]]
ids_to_delete

['5583866f-b4e6-47cd-9e36-2c559d60bbc3',
 '1c93dbe4-88de-41df-ad8a-935ec3152f09']

In [35]:
vector_store.delete(ids = ids_to_delete)
print('Deleted the ids')
for doc_id in ids_to_delete:
    print(doc_id)

Deleted the ids
5583866f-b4e6-47cd-9e36-2c559d60bbc3
1c93dbe4-88de-41df-ad8a-935ec3152f09


In [36]:
remaining_records = vector_store.get()
remaining_ids = remaining_records["ids"]

print(f"Remaining document count: {len(remaining_ids)}")
print("Remaining ids:")
for doc_id in remaining_ids:
    print(doc_id)

print("\nDeleted ids still present?")
for doc_id in ids_to_delete:
    print(f"{doc_id}: {doc_id in remaining_ids}")

Remaining document count: 8
Remaining ids:
384cf3e5-bab8-4b43-92f6-0d965be9fd71
b30f76ef-f6b3-42f6-a49f-aae9be25f336
8b4b5b54-9b70-4301-9023-7d62d98f789b
4439807e-3217-4d16-8e22-3e5cda6b1ef6
a80fd979-8d37-4c27-9fd8-fb52189ab165
b5678f4f-6ceb-4e84-a385-32f5e661f2ad
dfd3df89-ce06-4c4d-b384-be3a38e46b6d
b99df7ac-cbf3-4a77-acdd-d60f38e3b3c1

Deleted ids still present?
5583866f-b4e6-47cd-9e36-2c559d60bbc3: False
1c93dbe4-88de-41df-ad8a-935ec3152f09: False


In [37]:
print([doc.metadata["topic"] for doc in documents if doc.id in remaining_ids])

['AI', 'AI', 'AI', 'RAG', 'RAG', 'RAG', 'LLM', 'LLM']
